# torchvision.datasets.FashionMNIST EDA

## EDA의 목적
- FashionMNIST 데이터셋의 각 패션 아이템(예: 티셔츠, 바지, 스니커즈 등)을 조건부로 생성하는 작업을 수행하기에 앞서, 데이터의 형태와 분포를 확인하고, 최적의 모델 학습을 위한 인사이트를 획득한다. 
- 본 EDA를 통해 모델에 넣을 데이터 전처리 전략을 확정하고, 필요 시 데이터 전처리에 따른 모델 성능 변화 실험을 계획할 수 있다. 
- 각 섹션의 분석 결과를 정규화, 데이터 증강, 평가 방식에 대한 결정으로 연결하고, 7번 EDA 결론에서 결정 사항 표로 정리한다. 

## EDA 목차
> 각 섹션 끝에는 **인사이트 / 결정** 마크다운 셀을 두어, 분석 결과와 그에 따른 결정을 기록한다. 

0. 사전 설정
    - seed, 데이터 경로, 클래스 인덱스-이름 매핑, 그림 저장 경로를 정의한다. 
1. 데이터셋 훑어보기
    - **데이터셋 분할 건전성 확인** 
        - train, test split에 존재하는 데이터 개수와 클래스 라벨 절대도수, 비율을 확인한다.
            - 목적: 학습 시 조건 라벨을 균등하게 샘플링해도 되는지 확인한다. 
        - train, test split의 픽셀 통계(평균 밝기 등) 분포를 비교한다.
            - 목적: test split을 생성 성능 평가의 기준 분포(FID, 시각 비교)로 사용할 수 있는지 검토한다. 
    - **기본 sanity check**
        - 이미지 텐서의 shape, dtype, 픽셀 값 범위를 확인한다. 
    - **이미지 육안 확인**
        - train, test split 별로, 클래스마다 예시 이미지를 10장씩 그리드로 시각화한다. (seed 고정)
2. 픽셀 강도 분포
    - **전체 픽셀 분포**
        - 전체 픽셀 값 히스토그램(y축 log scale)을 그리고, 배경(0)과 최대값(255) 픽셀의 비율을 확인한다. 
    - **클래스별 이미지 단위 통계**
        - 클래스별로 이미지 단위 평균 밝기와 비배경(0이 아닌) 픽셀 비율의 분포를 violin plot으로 비교한다. 
            - 목적: 정규화 방식(`Normalize((0.5,), (0.5,))` + Generator의 Tanh 출력 vs mean/std 정규화)을 결정한다. 
3. 공간 구조
    - **클래스별 평균 / 표준편차 이미지**
        - 클래스별 평균 이미지와 픽셀별 표준편차 이미지를 시각화한다. 
    - **좌우 대칭성**
        - 클래스별 좌우 비대칭 점수(평균 이미지와 좌우 반전한 평균 이미지의 차이)를 비교한다. 
    - **객체 위치와 크기**
        - 클래스별 객체 bounding box, 무게중심, 점유 면적의 분포를 확인한다. 
            - 목적: 회전, 크롭, 좌우 반전, 밝기 조절 증강이 이 데이터에 적합한지 판단하고 증강 정책을 결정한다. 
            - 참고: GAN에서 real 이미지에만 증강을 적용하면, Generator가 증강된 분포를 학습하여 생성 이미지에 증강 흔적이 나타날 수 있다. 
4. 클래스 내 다양성과 클래스 간 유사도
    - **클래스 내 다양성**
        - 클래스별로 각 이미지와 클래스 평균 이미지 사이의 L2 거리 분포를 비교한다. 
    - **클래스 간 유사도**
        - 클래스 평균 이미지 간 코사인 유사도 히트맵을 그린다. 
        - 클래스별 일부 샘플을 PCA로 2차원에 투영하여 시각화한다. 
            - 목적: 서로 혼동되기 쉬운 클래스와 다양성이 큰 클래스를 찾아, 학습 중 집중적으로 모니터링할 클래스를 정한다. 
5. 데이터 품질
    - **중복 및 누수 확인**
        - train 내부의 중복 이미지와 train-test 간 중복 이미지(데이터 누수)를 확인한다. 
    - **이상치 확인**
        - 클래스별로 평균 이미지에서 가장 먼 이미지 top-k를 시각화한다. 
            - 목적: 라벨 노이즈나 이상치 의심 사례를 확인하고 제거 여부를 결정한다. 
6. 평가 기준선
    - 사전 준비: `uv add "torchmetrics[image]"`로 의존성을 추가한다. (`pyproject.toml`과 `uv.lock`은 함께 커밋)
    - **real-vs-real FID**
        - train 일부와 test 사이의 FID를 전체 및 클래스별로 계산한다. (Inception 입력을 위해 1채널 → 3채널 변환 필요)
            - 목적: 생성 모델 FID의 하한선(기준값)을 확보한다. 
    - **판정용 분류기**
        - 실데이터로 간단한 CNN 분류기를 학습하고 test 정확도를 측정한다. 
            - 목적: 생성 이미지가 조건 라벨대로 생성되었는지 판정할 분류기와 그 기준 정확도를 확보한다. 
7. EDA 결론
    - 2~6번의 결과를 정규화, 데이터 증강, 라벨 샘플링, 평가 방법, 모니터링 클래스 항목별 결정 사항 표로 정리한다. 

In [1]:
import os
import sys

import torch
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

# 실행 환경의 버전 확인 (팀원 간 환경 차이로 오류가 생겼을 때 원인 파악용)
print(f"Python      : {sys.version.split()[0]}")
print(f"torch       : {torch.__version__}")
print(f"torchvision : {torchvision.__version__}")
print(f"numpy       : {np.__version__}")
print(f"matplotlib  : {matplotlib.__version__}")

Python      : 3.12.9
torch       : 2.14.1
torchvision : 0.29.1
numpy       : 2.5.4
matplotlib  : 3.11.2


In [2]:
# 데이터셋을 저장할 경로 (없으면 자동 생성)
DATA_ROOT = '../data/raw'

# 랜덤 샘플 시각화 등의 재현성을 위한 seed 고정
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# 학습 데이터셋 (train=True) - 로컬에 없으면 다운로드
train_dataset = torchvision.datasets.FashionMNIST(
    root=DATA_ROOT,
    train=True,
    download=True,
)

# 테스트 데이터셋 (train=False)
test_dataset = torchvision.datasets.FashionMNIST(
    root=DATA_ROOT,
    train=False,
    download=True,
)

# 이미지: uint8 텐서 (N, 28, 28), 라벨: int64 텐서 (N,)
train_images, train_labels = train_dataset.data, train_dataset.targets
test_images, test_labels = test_dataset.data, test_dataset.targets

In [3]:
# 데이터 개수 확인
print(f"학습 데이터 개수  : {len(train_images):,}")
print(f"테스트 데이터 개수: {len(test_images):,}")

# 라벨 확인
print("클래스 라벨")
idx_to_class = {i: name for i, name in enumerate(train_dataset.classes)}
idx_to_class

학습 데이터 개수  : 60,000
테스트 데이터 개수: 10,000
클래스 라벨


{0: 'T-shirt/top',
 1: 'Trouser',
 2: 'Pullover',
 3: 'Dress',
 4: 'Coat',
 5: 'Sandal',
 6: 'Shirt',
 7: 'Sneaker',
 8: 'Bag',
 9: 'Ankle boot'}